In [1]:
# ================================================================
# RAG-BASED QUESTION ANSWERING SYSTEM
# Dataset: /kaggle/input/datasets/dtu25rit09abhinavpal/gen-ai-dataset
# ================================================================

import os
import sys
import fitz  # PyMuPDF
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from pathlib import Path
import json
from datetime import datetime

# ================================================================
# CONFIGURATION
# ================================================================

# UPDATE THIS PATH TO YOUR DATASET
PDF_FOLDER = "/kaggle/input/datasets/dtu25rit09abhinavpal/gen-ai-dataset"

# Text chunking parameters
CHUNK_SIZE = 150  # words per chunk
CHUNK_OVERLAP = 30  # word overlap between chunks
TOP_K = 3  # number of relevant chunks to retrieve

# Model for embeddings
MODEL_NAME = "all-MiniLM-L6-v2"  # Lightweight, fast model (384-dim embeddings)

# ================================================================
# UTILITY FUNCTIONS
# ================================================================

def check_and_list_pdfs(folder_path):
    """Check if folder exists and list all PDF files"""
    if not os.path.exists(folder_path):
        print(f"❌ ERROR: Folder not found: {folder_path}")
        return []
    
    pdf_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')])
    
    if not pdf_files:
        print(f"❌ WARNING: No PDF files found in {folder_path}")
        return []
    
    print(f"\n✅ Found {len(pdf_files)} PDF file(s):")
    for i, pdf in enumerate(pdf_files, 1):
        file_path = os.path.join(folder_path, pdf)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
        print(f"   [{i}] {pdf} ({file_size:.2f} MB)")
    
    return pdf_files


def extract_text_from_pdf(pdf_path):
    """Extract text from a single PDF file"""
    try:
        doc = fitz.open(pdf_path)
        full_text = ""
        
        for page_num, page in enumerate(doc):
            text = page.get_text().strip()
            if text:
                full_text += f"\n\n[PAGE {page_num + 1}]\n{text}"
        
        doc.close()
        return full_text
    except Exception as e:
        print(f"   ❌ Error reading {pdf_path}: {str(e)}")
        return ""


def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks"""
    words = text.split()
    chunks = []
    
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        if chunk.strip():  # Only add non-empty chunks
            chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks


def load_all_pdfs(folder_path, pdf_files):
    """Load all PDFs, extract text, and create chunks"""
    all_chunks = []
    chunk_sources = []
    chunk_metadata = []
    
    print("\n" + "=" * 60)
    print("📄 LOADING AND PROCESSING PDFs")
    print("=" * 60)
    
    total_text_length = 0
    
    for idx, pdf_file in enumerate(pdf_files, 1):
        full_path = os.path.join(folder_path, pdf_file)
        
        if not os.path.exists(full_path):
            print(f"\n❌ File not found: {pdf_file} — skipping")
            continue
        
        print(f"\n[{idx}/{len(pdf_files)}] Processing: {pdf_file}")
        
        # Extract text
        raw_text = extract_text_from_pdf(full_path)
        
        if not raw_text:
            print(f"   ⚠️  No text extracted from {pdf_file}")
            continue
        
        total_text_length += len(raw_text)
        
        # Split into chunks
        chunks = split_into_chunks(raw_text)
        
        # Store chunks and metadata
        all_chunks.extend(chunks)
        chunk_sources.extend([pdf_file] * len(chunks))
        chunk_metadata.extend([{
            'pdf': pdf_file,
            'chunk_index': i,
            'total_chunks_in_pdf': len(chunks)
        } for i in range(len(chunks))])
        
        print(f"   ✅ {len(chunks)} chunks extracted")
        print(f"   📊 Text length: {len(raw_text)} characters")
    
    print("\n" + "=" * 60)
    print(f"✅ TOTAL CHUNKS: {len(all_chunks)}")
    print(f"✅ Total PDFs processed: {len(set(chunk_sources))}")
    print(f"✅ Total text length: {total_text_length:,} characters")
    print("=" * 60 + "\n")
    
    return all_chunks, chunk_sources, chunk_metadata


# ================================================================
# EMBEDDING & FAISS INDEX
# ================================================================

class RAGSystem:
    """Main RAG System Class"""
    
    def __init__(self, chunks, chunk_sources, chunk_metadata, model_name=MODEL_NAME, top_k=TOP_K):
        self.chunks = chunks
        self.chunk_sources = chunk_sources
        self.chunk_metadata = chunk_metadata
        self.top_k = top_k
        self.model = None
        self.index = None
        self.embeddings = None
        
        print("\n" + "=" * 60)
        print("🤖 INITIALIZING RAG SYSTEM")
        print("=" * 60)
        
        # Load model
        self._load_model(model_name)
        
        # Generate embeddings
        self._generate_embeddings()
        
        # Build FAISS index
        self._build_faiss_index()
        
        print("\n✅ RAG System Ready!\n")
    
    def _load_model(self, model_name):
        """Load embedding model"""
        print(f"\n📥 Loading embedding model: {model_name}")
        try:
            self.model = SentenceTransformer(model_name)
            print(f"✅ Model loaded successfully")
            print(f"   Embedding dimension: 384")
        except Exception as e:
            print(f"❌ Error loading model: {str(e)}")
            sys.exit(1)
    
    def _generate_embeddings(self):
        """Generate embeddings for all chunks"""
        print(f"\n🔢 Generating embeddings for {len(self.chunks)} chunks...")
        
        try:
            self.embeddings = self.model.encode(
                self.chunks,
                show_progress_bar=True,
                convert_to_numpy=True
            )
            self.embeddings = np.array(self.embeddings).astype("float32")
            print(f"✅ Embeddings generated")
            print(f"   Shape: {self.embeddings.shape}")
        except Exception as e:
            print(f"❌ Error generating embeddings: {str(e)}")
            sys.exit(1)
    
    def _build_faiss_index(self):
        """Build FAISS index for similarity search"""
        print(f"\n🔍 Building FAISS index...")
        
        try:
            # Create L2 (Euclidean) distance index
            self.index = faiss.IndexFlatL2(self.embeddings.shape[1])
            self.index.add(self.embeddings)
            
            print(f"✅ FAISS index built successfully")
            print(f"   Total vectors stored: {self.index.ntotal}")
        except Exception as e:
            print(f"❌ Error building FAISS index: {str(e)}")
            sys.exit(1)
    
    def ask(self, query):
        """Ask a question and retrieve answers"""
        if not query.strip():
            print("❌ Query cannot be empty")
            return
        
        try:
            # Encode query
            query_vec = self.model.encode([query]).astype("float32")
            
            # Search FAISS index
            distances, indices = self.index.search(query_vec, self.top_k)
            
            # Display results
            print("\n" + "═" * 70)
            print(f"❓ QUERY: {query}")
            print("═" * 70)
            
            for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
                print(f"\n🔹 [{rank}] RELEVANCE SCORE: {distance:.4f}")
                print(f"    📄 Source: {self.chunk_sources[idx]}")
                print(f"    📍 Chunk #{self.chunk_metadata[idx]['chunk_index'] + 1}")
                print(f"    📝 Text Preview:")
                print(f"    {self.chunks[idx][:400]}...")
            
            print("\n" + "═" * 70 + "\n")
            
            return {
                'query': query,
                'results': [
                    {
                        'rank': rank,
                        'source': self.chunk_sources[idx],
                        'distance': float(distance),
                        'text': self.chunks[idx],
                        'metadata': self.chunk_metadata[idx]
                    }
                    for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1)
                ]
            }
        
        except Exception as e:
            print(f"❌ Error during search: {str(e)}")
            return None
    
    def batch_ask(self, queries):
        """Ask multiple questions"""
        results = []
        for query in queries:
            result = self.ask(query)
            if result:
                results.append(result)
        return results
    
    def get_system_stats(self):
        """Get system statistics"""
        return {
            'total_chunks': len(self.chunks),
            'total_pdfs': len(set(self.chunk_sources)),
            'embedding_dimension': self.embeddings.shape[1],
            'index_vectors': self.index.ntotal,
            'model_name': 'all-MiniLM-L6-v2'
        }


# ================================================================
# MAIN EXECUTION
# ================================================================

def main():
    """Main function to run the RAG system"""
    
    print("\n" + "🎯" * 35)
    print(" RAG-BASED QUESTION ANSWERING SYSTEM ")
    print("🎯" * 35 + "\n")
    
    # Step 1: List available PDFs
    pdf_files = check_and_list_pdfs(PDF_FOLDER)
    
    if not pdf_files:
        print("\n❌ No PDFs found. Exiting.")
        return
    
    # Step 2: Load and process PDFs
    chunks, chunk_sources, chunk_metadata = load_all_pdfs(PDF_FOLDER, pdf_files)
    
    if not chunks:
        print("\n❌ No chunks created. Exiting.")
        return
    
    # Step 3: Initialize RAG System
    rag = RAGSystem(chunks, chunk_sources, chunk_metadata, top_k=TOP_K)
    
    # Step 4: Display system statistics
    stats = rag.get_system_stats()
    print("\n📊 SYSTEM STATISTICS:")
    print(f"   ├─ Total Chunks: {stats['total_chunks']}")
    print(f"   ├─ Total PDFs: {stats['total_pdfs']}")
    print(f"   ├─ Embedding Dimension: {stats['embedding_dimension']}")
    print(f"   ├─ Index Vectors: {stats['index_vectors']}")
    print(f"   └─ Model: {stats['model_name']}\n")
    
    # Step 5: Run sample queries
    sample_queries = [
        "Types of Generative Ai?",
        "Explain GEN Ai?",
        "Explain RAG?",
        "Application Of generative Ai?",
    ]
    
    print("\n" + "=" * 70)
    print("🚀 RUNNING SAMPLE QUERIES")
    print("=" * 70)
    
    for query in sample_queries:
        rag.ask(query)
    
    # Step 6: Interactive mode
    print("\n" + "=" * 70)
    print("💬 INTERACTIVE MODE")
    print("=" * 70)
    print("\nYou can now ask questions about the documents.")
    print("Type 'exit' to quit.\n")
    
    while True:
        try:
            user_query = input("❓ Ask a question: ").strip()
            
            if user_query.lower() == 'exit':
                print("\n👋 Goodbye!")
                break
            
            if not user_query:
                continue
            
            rag.ask(user_query)
        
        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'fitz'